In [10]:
# Cell 1 — Imports
import sys
import os

# Walk upwards from the current working directory until we find the
# folder that contains 'backend' (the project root), then add it to sys.path
candidate = os.getcwd()
for _ in range(5):
    if os.path.isdir(os.path.join(candidate, 'backend')):
        break
    candidate = os.path.dirname(candidate)

project_root = candidate
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print('cwd          :', os.getcwd())
print('project_root :', project_root)
print('backend dir exists:', os.path.isdir(os.path.join(project_root, 'backend')))

from backend.pipeline.loader import load_input
from backend.pipeline.preprocessor import preprocess
from backend.pipeline.embedder import embed
from backend.pipeline.vector_store import build_and_save, load
from backend.pipeline.retriever import retrieve
from backend.pipeline.generator import generate

print('All pipeline modules imported successfully')

cwd          : c:\Users\dahab\OneDrive - European Universities in Egypt\University\CS Year 3\Sem 2\Final Project\Student Assistant\notebooks
project_root : c:\Users\dahab\OneDrive - European Universities in Egypt\University\CS Year 3\Sem 2\Final Project\Student Assistant
backend dir exists: True
All pipeline modules imported successfully


In [34]:
# Cell 2 — Dataset: load one sample from DocVQA
from datasets import load_dataset, Dataset

ds = load_dataset('lmms-lab/DocVQA', 'DocVQA', split='validation', streaming=True)
ds = Dataset.from_list(list(ds.take(10)))

sample = ds[5]
image = sample['image']

print('Question:', sample['question'])
print('Answers :', sample['answers'])
print('Image   :', image)

[loader] HTTP Request: HEAD https://huggingface.co/datasets/lmms-lab/DocVQA/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
[loader] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/lmms-lab/DocVQA/539088ef8a8ada01ac8e2e6d4e372586748a265e/README.md "HTTP/1.1 200 OK"
[loader] HTTP Request: HEAD https://huggingface.co/datasets/lmms-lab/DocVQA/resolve/539088ef8a8ada01ac8e2e6d4e372586748a265e/DocVQA.py "HTTP/1.1 404 Not Found"
[loader] HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/lmms-lab/DocVQA/lmms-lab/DocVQA.py "HTTP/1.1 404 Not Found"
[loader] HTTP Request: HEAD https://huggingface.co/datasets/lmms-lab/DocVQA/resolve/539088ef8a8ada01ac8e2e6d4e372586748a265e/.huggingface.yaml "HTTP/1.1 404 Not Found"
[loader] HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=lmms-lab/DocVQA "HTTP/1.1 200 OK"
[loader] HTTP Request: HEAD https://huggingface.co/datasets/lmms-lab/DocVQA/resolve/539088ef8a8ada01ac8e2e6d

Question: What the location address of NSDA?
Answers : ['1128 SIXTEENTH ST., N. W., WASHINGTON, D. C. 20036', '1128 sixteenth st., N. W., washington, D. C. 20036']
Image   : <PIL.PngImagePlugin.PngImageFile image mode=L size=1370x1480 at 0x1DA26A29C50>


In [35]:
# Cell 3 — Load: run OCR on the image via loader.py
image_path = '../data/sample/docvqa_sample.png'
image.save(image_path)

raw_text = load_input(image_path, input_type='image')
print('Raw OCR text:')
print(raw_text)

[loader] Running OCR on image → ../data/sample/docvqa_sample.png
[loader] Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
[loader] OCR complete — 644 characters extracted


Raw OCR text:
The best thing
between two sandwiches:
Soft drinks
go
with
all
kinds
of
drinks contain the purest; filtered water.
sandwiches. Round ones, square ones,
So
sandwich
soft   drinks  among
fat ones and lean ones.
your
sandwiches.
And
celebrate
Na -
Not
only
do
quench  large
tional Sandwich
Month every month in
thirsts in
a fun way; they also help bal-
the year_
ance the diet:
After all, healthy bodies
information on soft drinks and
need 5 to
glasses of water a day
Soft
the balanced diet, write:
NATIONAL soFt drink ASSOCIATION
NSDA
1128 SIXTEENTH ST  N.
WASHINGTON; D_
20036
Source: https IIWW industrydocuments ucsf eduldocsiqqvf0227
they
For


In [36]:
# Cell 4 — Preprocess: clean and chunk the raw text
chunks = preprocess(raw_text)

print(f'Number of chunks: {len(chunks)}')
print('First chunk:')
print(chunks[0])

Number of chunks: 1
First chunk:
The best thing between two sandwiches: Soft drinks go with all kinds of drinks contain the purest; filtered water. sandwiches. Round ones, square ones, So sandwich soft drinks among fat ones and lean ones. your sandwiches. And celebrate Na - Not only do quench large tional Sandwich Month every month in thirsts in a fun way; they also help bal- the year_ ance the diet: After all, healthy bodies information on soft drinks and need 5 to glasses of water a day Soft the balanced diet, write: NATIONAL soFt drink ASSOCIATION NSDA 1128 SIXTEENTH ST N. WASHINGTON; D_ 20036 Source: https IIWW industrydocuments ucsf eduldocsiqqvf0227 they For


In [37]:
# Cell 5 — Embed and index: build the FAISS vector store
embeddings = embed(chunks)
build_and_save(embeddings, chunks)

index, stored_chunks = load()
print(f'Total vectors stored in FAISS index: {index.ntotal}')

Total vectors stored in FAISS index: 1


In [38]:
# Cell 6 — Query: hardcode the ground-truth question from the sample
query = sample['question']
print('Query:', query)

Query: What the location address of NSDA?


In [39]:
# Cell 7 — Retrieve: top 3 most relevant chunks
top_chunks = retrieve(query, index, stored_chunks, k=3)

print('Top retrieved chunks:')
for i, chunk in enumerate(top_chunks, start=1):
    print(f'[{i}] {chunk}')

Top retrieved chunks:
[1] The best thing between two sandwiches: Soft drinks go with all kinds of drinks contain the purest; filtered water. sandwiches. Round ones, square ones, So sandwich soft drinks among fat ones and lean ones. your sandwiches. And celebrate Na - Not only do quench large tional Sandwich Month every month in thirsts in a fun way; they also help bal- the year_ ance the diet: After all, healthy bodies information on soft drinks and need 5 to glasses of water a day Soft the balanced diet, write: NATIONAL soFt drink ASSOCIATION NSDA 1128 SIXTEENTH ST N. WASHINGTON; D_ 20036 Source: https IIWW industrydocuments ucsf eduldocsiqqvf0227 they For


In [40]:
# Cell 8 — Generate: produce an answer with flan-t5-large (CPU)
answer = generate(query, top_chunks)
print('Generated answer:', answer)

Generated answer: 1128 SIXTEENTH ST N. WASHINGTON


In [41]:
# Cell 9 — Evaluate: compare generated answer with ground truth
ground_truth = sample['answers'][0]

print('Generated answer :', answer)
print('Ground truth     :', ground_truth)

Generated answer : 1128 SIXTEENTH ST N. WASHINGTON
Ground truth     : 1128 SIXTEENTH ST., N. W., WASHINGTON, D. C. 20036
